In [ ]:
#conda activate burnseverity
import pandas as pd
import numpy as np
from matplotlib import pyplot as plt

import requests
import json

import geopandas as gpd

import rasterio as rio
from rasterio.warp import calculate_default_transform, reproject, Resampling
from rasterio.mask import mask
from rasterio import features
from rasterio.plot import show_hist

from shapely.geometry import shape, mapping
from shapely.ops import unary_union

import validation as val


### Validation goals

#### Questions
1. Difference between our dNBR and dNBR from BAIR?
2. Are the two different indices truly different in our tool and is ours truly more sensitive?
3. Do our boundaries and those from CalFire match?
4. Sensitivity around chosen time windows?

#### Metrics
1. Recoarse Sentinel to Landsat, $R^2$
2. $var(dNBR) < var(RBR)$
3. Percent overlap
4. Before period fixed through three weeks. After ${5, 10, 15, 21, 30, 45, 60, 90}$. Criteria: stable and low variance 

In [6]:
fires = pd.read_csv('fire_processing_jobs.csv')
calfire = gpd.read_file('Validation_Fire_Perimeters_2015_2024.shp')

In [ ]:
def process_fire_metrics(fire_name, fire_days, fires, calfire):
    """
    Process fire severity metrics for a given fire and post-fire days.
    
    Returns:
    - list of dicts (one per metric: dnbr, rdnbr), or False if job not complete/error
    """
    
    # Get fire_event_name and job_id
    fire_rows = fires.loc[(fires['fire_name'] == fire_name) & (fires['post_fire_days'] == fire_days)]
    if len(fire_rows) == 0:
        print(f"  Warning: No job found for {fire_name} at {fire_days} days")
        return False
    
    fire_event_name = fire_rows['fire_event_name'].values[0]
    job_id = fire_rows['job_id'].values[0]
    
    # Get fire polygon
    fire_polygon = calfire[calfire['FIRE_NAME'] == fire_name]
    if len(fire_polygon) == 0:
        print(f"  Warning: No CalFire boundary found for {fire_name}")
        return False
    
    # Get URLs from API
    try:
        request = requests.get(f"https://fire-recovery-backend-dev-113009620257.us-central1.run.app/fire-recovery/result/analyze_fire_severity/{fire_event_name}/{job_id}")
        response_data = request.json()
        
        if response_data.get('status') != 'complete':
            print(f"  Job {fire_event_name} not completed yet.")
            return False
        
        urls = response_data.get('coarse_severity_cog_urls')
        dnbr_url = urls.get('dnbr')
        rdnbr_url = urls.get('rdnbr')
    except Exception as e:
        print(f"  Error fetching URLs: {e}")
        return False
    
    # Store results - one row per metric
    results = []
    
    # Process both dNBR and RdNBR
    for metric_name, metric_url in [('dnbr', dnbr_url), ('rdnbr', rdnbr_url)]:
        try:
            with rio.open(metric_url) as src:
                # Reproject fire polygon to raster CRS
                fire_poly_reproj = fire_polygon.to_crs(src.crs)
                
                # Crop to CalFire boundary
                geoms = [mapping(geom) for geom in fire_poly_reproj.geometry]
                cropped, cropped_transform = mask(src, geoms, crop=True, nodata=np.nan)
                cropped_data = cropped[0]
                
                # Remove nodata values
                valid_data = cropped_data[~np.isnan(cropped_data)]
                
                # CalFire metrics (all pixels in boundary)
                calfire_mean = np.mean(valid_data) if len(valid_data) > 0 else np.nan
                calfire_var = np.var(valid_data) if len(valid_data) > 0 else np.nan
                
                # Filtered metrics (pixels > 0)
                filtered_data = valid_data[valid_data > 0]
                filtered_mean = np.mean(filtered_data) if len(filtered_data) > 0 else np.nan
                filtered_var = np.var(filtered_data) if len(filtered_data) > 0 else np.nan
                
                # Create polygon from filtered pixels (pixels > 0)
                filtered_mask = cropped_data > 0
                shapes_gen = features.shapes(
                    filtered_mask.astype(np.int16),
                    mask=filtered_mask,
                    transform=cropped_transform
                )
                
                # Combine all polygons
                filtered_polygons = [shape(geom) for geom, val in shapes_gen if val == 1]
                
                if len(filtered_polygons) > 0:
                    filtered_polygon = unary_union(filtered_polygons)
                    filtered_gdf = gpd.GeoDataFrame([1], geometry=[filtered_polygon], crs=src.crs)
                    filtered_polygon_latlon = filtered_gdf.to_crs(epsg=4326).geometry.values[0]
                else:
                    filtered_polygon_latlon = None
                
                # Append row for this metric
                results.append({
                    'fire_name': fire_name,
                    'fire_days': fire_days,
                    'fire_event_name': fire_event_name,
                    'metric': metric_name,
                    'calfire_mean': calfire_mean,
                    'calfire_var': calfire_var,
                    'filtered_mean': filtered_mean,
                    'filtered_var': filtered_var,
                    'filtered_polygon': filtered_polygon_latlon
                })
                
        except Exception as e:
            print(f"  Error processing {metric_name}: {e}")
            return False
    
    return results

In [ ]:
# Process all fires and date ranges
results_list = []
current_fire = None

for _, row in fires.iterrows():
    # Print only when starting a new fire
    if row['fire_name'] != current_fire:
        print(f"Processing {row['fire_name']}...")
        current_fire = row['fire_name']
    
    result = process_fire_metrics(row['fire_name'], row['post_fire_days'], fires, calfire)
    
    if result is False:
        continue
    
    # result is now a list of dicts (one per metric)
    results_list.extend(result)

# Create DataFrame
metrics_df = pd.DataFrame(results_list)

# Save as spatial file (GeoPackage format works well with R's sf package)
metrics_gdf = gpd.GeoDataFrame(
    metrics_df,
    geometry='filtered_polygon',
    crs='EPSG:4326'
)

# Save as GeoPackage (preferred for R/sf)
metrics_gdf.to_file('validation_metrics.gpkg', driver='GPKG')

# Also save attributes as CSV for quick viewing
metrics_df.drop('filtered_polygon', axis=1).to_csv('validation_metrics.csv', index=False)

print(f"\nFinal table shape: {metrics_df.shape}")
print("Saved spatial data to validation_metrics.gpkg")
print("Saved CSV to validation_metrics.csv")

metrics_df.head()

Processing COFFEE POT_2024-08-03_5...
Processing COFFEE POT_2024-08-03_10...
Processing COFFEE POT_2024-08-03_15...
Processing COFFEE POT_2024-08-03_21...
Processing COFFEE POT_2024-08-03_30...
Processing COFFEE POT_2024-08-03_45...
Processing COFFEE POT_2024-08-03_60...
Processing COFFEE POT_2024-08-03_90...
Processing SENTINEL_2024-07-14_5...
Processing SENTINEL_2024-07-14_10...
Processing SENTINEL_2024-07-14_15...
Processing SENTINEL_2024-07-14_21...
Processing SENTINEL_2024-07-14_30...
Processing SENTINEL_2024-07-14_45...
Processing SENTINEL_2024-07-14_60...
Processing SENTINEL_2024-07-14_90...
Processing SIMPSON_2024-07-26_5...
Processing SIMPSON_2024-07-26_10...
Processing SIMPSON_2024-07-26_15...
Processing SIMPSON_2024-07-26_21...
Processing SIMPSON_2024-07-26_30...
Processing SIMPSON_2024-07-26_45...
Processing SIMPSON_2024-07-26_60...
Processing SIMPSON_2024-07-26_90...
Processing YORK_2023-07-28_5...
  Job YORK_2023-07-28_5 not completed yet.
Processing YORK_2023-07-28_10...

/opt/miniconda3/envs/burnseverity/lib/python3.13/site-packages/numpy/_core/_methods.py:188: RuntimeWarning: invalid value encountered in subtract
  x = um.subtract(arr, arrmean, out=...)
/opt/miniconda3/envs/burnseverity/lib/python3.13/site-packages/numpy/_core/_methods.py:188: RuntimeWarning: invalid value encountered in subtract
  x = um.subtract(arr, arrmean, out=...)


Processing GEOLOGY_2023-06-10_5...
Processing GEOLOGY_2023-06-10_10...
Processing GEOLOGY_2023-06-10_15...
Processing GEOLOGY_2023-06-10_21...
Processing GEOLOGY_2023-06-10_30...
Processing GEOLOGY_2023-06-10_45...
Processing GEOLOGY_2023-06-10_60...
Processing GEOLOGY_2023-06-10_90...
Processing VALLEY_2023-06-17_5...
Processing VALLEY_2023-06-17_10...
Processing VALLEY_2023-06-17_15...
Processing VALLEY_2023-06-17_21...
Processing VALLEY_2023-06-17_30...
Processing VALLEY_2023-06-17_45...
Processing VALLEY_2023-06-17_60...
Processing VALLEY_2023-06-17_90...
Processing SYCAMORE_2023-10-16_5...
Processing SYCAMORE_2023-10-16_10...
Processing SYCAMORE_2023-10-16_15...
Processing SYCAMORE_2023-10-16_21...
Processing SYCAMORE_2023-10-16_30...
Processing SYCAMORE_2023-10-16_45...
Processing SYCAMORE_2023-10-16_60...
Processing SYCAMORE_2023-10-16_90...
Processing SUMMIT_2022-08-03_5...
Processing SUMMIT_2022-08-03_10...
Processing SUMMIT_2022-08-03_15...
Processing SUMMIT_2022-08-03_21...


,fire_name,fire_days,fire_event_name,boundary_mean_dnbr,boundary_var_dnbr,burned_mean_dnbr,burned_var_dnbr,boundary_mean_rdnbr,boundary_var_rdnbr,burned_mean_rdnbr,burned_var_rdnbr,burned_polygon
0,COFFEE POT,5,COFFEE POT_2024-08-03_5,0.058276,0.008079,0.076561,0.007665,0.064776,0.007889,0.086878,0.005868,MULTIPOLYGON (((-118.78054641229755 36.3526655...
1,COFFEE POT,10,COFFEE POT_2024-08-03_10,0.055370,0.007539,0.074775,0.006698,0.065616,0.008669,0.089038,0.006691,MULTIPOLYGON (((-118.7948966008226 36.35266556...
2,COFFEE POT,15,COFFEE POT_2024-08-03_15,0.028381,0.009027,0.064690,0.006796,0.034460,0.010358,0.078227,0.006366,MULTIPOLYGON (((-118.79507597817916 36.3524830...
3,COFFEE POT,21,COFFEE POT_2024-08-03_21,-0.040396,0.014176,0.061458,0.009702,-0.043407,0.018009,0.077438,0.009087,MULTIPOLYGON (((-118.79884290266698 36.3564982...
4,COFFEE POT,30,COFFEE POT_2024-08-03_30,0.011391,0.020376,0.093045,0.013553,0.015745,0.032647,0.115884,0.016674,MULTIPOLYGON (((-118.80039730428956 36.3589198...
